In [33]:
from IPython.core.display import display, Javascript

def wrap_text():
    display(Javascript('''
        function changeCss(cssFile) {
            let head = document.getElementsByTagName("head")[0];
            let link = document.createElement("link");
            link.rel = "stylesheet";
            link.type = "text/css";
            link.href = cssFile;
            head.appendChild(link);
        }
        changeCss("https://raw.githubusercontent.com/colab-resources/colab-text-wrap/main/colab-text-wrap.css");
    '''))

wrap_text()


<IPython.core.display.Javascript object>

In [4]:
! pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 3.8 MB/s eta 0:00:00


In [16]:
! pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.3 MB/s eta 0:00:00


In [25]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 25.2 MB/s eta 0:00:00


In [50]:
from docx import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [18]:
from google.colab import files
uploaded = files.upload()

Saving solar panel technology.docx to solar panel technology.docx


In [48]:
def extract_text_from_docx(file_path):
    doc = Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    return text

text_data = extract_text_from_docx("solar panel technology.docx")
print(text_data[:500])

New Solar Panel Technology Trends Shaping the Future
Efficiency Skyrockets With New Solar Panel Technologies
Solar panel efficiency has seen remarkable advancements over the past two to three decades. In the early days, solar panels had a conversion efficiency of around 10%, meaning they could only convert about a tenth of the sunlight they captured into usable electricity. However, solar panel efficiency rates have increased dramatically thanks to continuous research, development, and technolog


In [45]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
def split_text(text, chunk_size=300):
    """Splits text into smaller chunks for vector indexing."""
    sentences = text.split(". ")
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

chunks = split_text(text_data)
embeddings = model.encode(chunks)


In [46]:
chunks[0]

'New Solar Panel Technology Trends Shaping the Future\nEfficiency Skyrockets With New Solar Panel Technologies\nSolar panel efficiency has seen remarkable advancements over the past two to three decades.'

In [47]:
import pickle
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

In [22]:
embeddings.shape

(74, 384)

In [26]:
import faiss
import numpy as np

vector_dim = embeddings.shape[1]

index = faiss.IndexFlatL2(vector_dim)
index.add(np.array(embeddings))
faiss.write_index(index, "solar_vectors.index")


In [27]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7be922b5b0c0> >

In [28]:
index = faiss.read_index("solar_vectors.index")

def retrieve_relevant_text(query, top_k=1):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    return [chunks[i] for i in indices[0]]



In [51]:
llm=ChatGroq(model="mixtral-8x7b-32768",temperature=0.2)

In [41]:
def generate_response(user_query):
    """Fetch relevant data from FAISS and pass to LLM."""
    retrieved_text = retrieve_relevant_text(user_query, top_k=2)
    print(f"Retrieved Text: {len(retrieved_text)}")
    # Create prompt with retrieved text
    system_message = "You are an intelligent assistant that provides accurate, helpful information about solar energy based on the information provided(if not, answer according to your knowledge)."
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", f"Use the following information to answer: {retrieved_text} \n\nUser Query: {user_query}")
    ])

    # Generate response
    chain = prompt_template | llm
    response = chain.invoke({"text": user_query})

    return response.content

# Example Usage
user_input = "What is the installation process in detail"
answer = generate_response(user_input)
print(answer)

Retrieved Text: 2
Based on the information provided, here is a detailed solar panel installation process:

1. Obtain Permits and Grants:
   - Finalize your installation plan.
   - Obtain necessary permits to legalize the installation.
   - Explore potential subsidies or incentives for solar panel installation available in your area.

2. Install Racking System:
   - For Pitched Roofs - Use flashings, screws, and bolts to attach rails by drilling the roof. Apply sealant to prevent water leakage.
   - For Flat Roofs - Installers use either ballasts or concrete blocks to rack the solar panels.
   - Secure the structures into place properly.
   - Install the racking rails.

3. Install Solar Panels:
   - Place solar panels onto the racking system.
   - Connect the panels to the racking system using clamps.
   - Ensure that the panels are level and secure.

4. Install Conduit and Wiring:
   - Run conduit from the solar array to the inverter location.
   - Connect the panels using MC4 (Multi-C